# Atlas Raman — Inference Demo

End-to-end demonstration of the production Stage 15F classifier: drop in a raw `.xls` Raman file, get a predicted class + probabilities.

Uses `atlas.inference.predict_from_xls()` and `predict_from_array()`.

**Model:** sklearn Pipeline — StandardScaler + LogisticRegression-L2 — trained on 35 MI-selected engineered features extracted from 87 confocal Raman files (7,122 QC-passed pixels).

**Classes:** STEC | Non-STEC | Salmonella | H2O

## How to run

From the repo root (worktree or main checkout), create the required symlinks before launching Jupyter:

```bash
ln -s /path/to/NomadX/data_cache   data_cache
ln -s /path/to/NomadX/artifacts    artifacts
ln -s "/path/to/NomadX/Atlas Data" "Atlas Data"   # quotes required — space in name
ln -s /path/to/NomadX/.venv        .venv

export OMP_NUM_THREADS=1 MKL_NUM_THREADS=1 OPENBLAS_NUM_THREADS=1
.venv/bin/jupyter lab
```

The `"Atlas Data"` symlink (with quotes) is **required** for Unit 11 — it is the only notebook that reads raw `.xls` files directly.

## Inference pipeline overview

```
raw .xls file
     |
  1. parse_file()
     Reads the raw Atlas .xls tab-delimited format.
     Extracts x/y pixel coordinates + intensity matrix (N_pixels x N_bins).
     |
  2. preprocess_matrix()
     QC (SNR / cosmic-ray rejection) + arPLS baseline + Savitzky-Golay
     smoothing + SNV normalization. Crops to canonical 987-bin wavenumber axis.
     |
  3. Per-pixel feature extraction
     - Band Pseudo-Voigt fits (166 features: center, FWHM, amplitude, area
       for key Raman bands)
     - Spectral DWT coefficients + PCA projections onto frozen ROI axes
     - SAM (Spectral Angle Mapper) scores vs frozen class templates
     - MCR-ALS concentrations (K=7 components, frozen global model)
     - Spatial moments (skewness, kurtosis of intensity maps)
     |
  4. File-level aggregation
     Mean-pool all features across QC-passed pixels -> 1 row per file.
     |
  5. Feature subsetting
     Select the 35 production features chosen by per-fold mutual information.
     |
  6. StandardScaler + LogisticRegression-L2
     Predict class label + 4-class probability vector.
```

In [1]:
import os
import sys
from pathlib import Path

# Thread limits — must be set before numpy import
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'

# Locate the NomadX main repo root that contains atlas/inference.py.
# The worktree's own atlas/ is a frozen copy without inference.py, so we walk
# up the tree looking for the directory that has both atlas/inference.py and artifacts/.
REPO_ROOT = None
for candidate in [Path('/Users/devashishthapliyal/Documents/NomadX')] + [
    p for p in Path.cwd().parents
] + [Path.cwd()]:
    if (candidate / 'atlas' / 'inference.py').exists() and (candidate / 'artifacts').exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise RuntimeError('Cannot find NomadX repo root with atlas/inference.py. '
                       'Check symlinks and run from the worktree root.')

print(f'Repo root: {REPO_ROOT}')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ATLAS_DATA = REPO_ROOT / 'Atlas Data'
DATA_CACHE = REPO_ROOT / 'data_cache'
ARTIFACTS  = REPO_ROOT / 'artifacts'

print(f'Atlas Data exists: {ATLAS_DATA.exists()}')
print(f'data_cache exists: {DATA_CACHE.exists()}')
print(f'artifacts  exists: {ARTIFACTS.exists()}')

Repo root: /Users/devashishthapliyal/Documents/NomadX


Atlas Data exists: True
data_cache exists: True
artifacts  exists: True


## Section 4 — Pick representative files (one per class)

In [2]:
# Representative files — one from each of the four classes.
DEMO_FILES = {
    'STEC':       ATLAS_DATA / 'STEC'      / 'O103H2'  / 'R359_100_10000ms_260226.xls',
    'Salmonella': ATLAS_DATA / 'Salmonella' / 'Dublin'  / 'R370_100_10000ms_260305.xls',
    'Non-STEC':   ATLAS_DATA / 'Non STEC'  / '83972'   / 'R475_100_10000ms_260317.xls',
    'H2O':        ATLAS_DATA / 'H20'        / 'R372_100_10000ms_260306.xls',
}

print('Demo files selected:')
for label, path in DEMO_FILES.items():
    status = 'OK' if path.exists() else 'MISSING'
    print(f'  [{status}]  {label:12s}  {path.name}')

Demo files selected:
  [OK]  STEC          R359_100_10000ms_260226.xls
  [OK]  Salmonella    R370_100_10000ms_260305.xls
  [OK]  Non-STEC      R475_100_10000ms_260317.xls
  [OK]  H2O           R372_100_10000ms_260306.xls


## Section 5 — predict_from_xls on each demo file

In [3]:
from atlas.inference import predict_from_xls, predict_from_array

print('predict_from_xls results')
print('=' * 70)

results = {}
for true_label, path in DEMO_FILES.items():
    result = predict_from_xls(path)
    results[true_label] = result

    predicted = result['class']
    proba = result['probabilities']
    sorted_proba = sorted(proba.items(), key=lambda x: x[1], reverse=True)
    top2 = sorted_proba[:2]

    marker = 'CORRECT' if predicted == true_label else 'WRONG'
    print(f'\nFile:        {path.name}')
    print(f'True class:  {true_label}')
    print(f'Predicted:   {predicted}  [{marker}]')
    print(f'Top-2 probs: {top2[0][0]}={top2[0][1]:.3f}  |  {top2[1][0]}={top2[1][1]:.3f}')

print('\n' + '=' * 70)
print('predict_from_xls calls complete.')

predict_from_xls results



File:        R359_100_10000ms_260226.xls
True class:  STEC
Predicted:   STEC  [CORRECT]
Top-2 probs: STEC=0.875  |  Salmonella=0.093



File:        R370_100_10000ms_260305.xls
True class:  Salmonella
Predicted:   STEC  [WRONG]
Top-2 probs: STEC=0.345  |  Salmonella=0.252



File:        R475_100_10000ms_260317.xls
True class:  Non-STEC
Predicted:   Non-STEC  [CORRECT]
Top-2 probs: Non-STEC=0.750  |  Salmonella=0.158



File:        R372_100_10000ms_260306.xls
True class:  H2O
Predicted:   H2O  [CORRECT]
Top-2 probs: H2O=0.981  |  STEC=0.017

predict_from_xls calls complete.


## Section 6 — Detailed view for one file

In [4]:
# Use the STEC result for the detailed breakdown
detail_label = 'STEC'
detail_result = results[detail_label]

# Key band annotations (wavenumber cm-1 -> label)
KEY_BANDS = {
    1004: 'Phe (aa 1004)',
    1117: 'LPS (1117)',
    1242: 'Amide III (1242)',
    1454: 'Lipid CH2 (1454)',
    1658: 'Amide I (1658)',
}

wn   = detail_result['wn']
spec = detail_result['spectrum_mean']
proba = detail_result['probabilities']
feats = detail_result['feature_values']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: mean preprocessed spectrum with band markers
ax = axes[0]
ax.plot(wn, spec, lw=0.9, color='steelblue')
ax.set_xlabel('Wavenumber (cm$^{-1}$)')
ax.set_ylabel('SNV-normalised intensity (a.u.)')
ax.set_title(f'Mean preprocessed spectrum\n({detail_label} — {DEMO_FILES[detail_label].name})')
ax.set_xlim(float(wn.min()), float(wn.max()))

spec_max = float(spec.max())
for band_cm, band_label in KEY_BANDS.items():
    idx = int(np.argmin(np.abs(wn - band_cm)))
    ax.axvline(float(wn[idx]), color='tomato', lw=0.8, linestyle='--', alpha=0.7)
    ax.text(float(wn[idx]), spec_max * 0.82,
            band_label, ha='center', va='bottom', fontsize=6.5, color='tomato',
            rotation=90)

# Right: 4-class probability bar chart
ax2 = axes[1]
class_order = ['STEC', 'Non-STEC', 'Salmonella', 'H2O']
proba_vals = [proba.get(c, 0.0) for c in class_order]
colors = ['#e74c3c', '#3498db', '#2ecc71', '#95a5a6']
bars = ax2.bar(class_order, proba_vals, color=colors, edgecolor='black', linewidth=0.5)
ax2.set_ylim(0, 1.1)
ax2.set_ylabel('Predicted probability')
ax2.set_title(f'Class probabilities\nPredicted: {detail_result["class"]}')
for bar, val in zip(bars, proba_vals):
    ax2.text(bar.get_x() + bar.get_width() / 2, val + 0.02,
             f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('/tmp/atlas_demo_spectrum.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot saved to /tmp/atlas_demo_spectrum.png')

# Table of the 35 production feature values
print(f'\nProduction feature values (35 features)  |  {detail_label} file')
print('-' * 55)
feat_df = pd.DataFrame(
    [{'Feature': k, 'Value': f'{v:.6f}'} for k, v in feats.items()]
)
feat_df.index = range(1, len(feat_df) + 1)
print(feat_df.to_string())

Plot saved to /tmp/atlas_demo_spectrum.png

Production feature values (35 features)  |  STEC file
-------------------------------------------------------
                      Feature        Value
1          roi_ch_stretch_std     0.030683
2     fit_amide_iii_1242_area     2.162233
3             roi_silent_kurt     1.593934
4   fit_amide_iii_1242_height     0.049354
5             d1_auc_lps_1117     0.036578
6           d1_auc_lipid_1454     0.033768
7         d1_auc_amide_i_1658     0.020905
8       fit_lipid_1454_height     0.112731
9     fit_amide_iii_1242_rmse     0.001996
10        fit_lipid_1454_area     4.324514
11        fit_lps_1117_height    -0.073193
12          fit_lps_1117_area    -4.409027
13          fit_lps_1117_fwhm    38.556683
14          d2_auc_lipid_1454     0.006823
15            d2_auc_lps_1117     0.005152
16      roi_lps_chain_entropy     5.379728
17        fit_lps_1194_height    -0.009515
18            roi_silent_skew     0.915090
19             d2_auc_aa_1004

/var/folders/6w/9kd9pqy15f55_v_jbsnrkdx00000gn/T/ipykernel_39687/1006991638.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 7 — predict_from_array demo + cross-check

In [5]:
# Load the preprocessed spectra cache
X_all = np.load(DATA_CACHE / 'spectra_array_preprocessed.npy')    # (7999, 987)
wn_pp = np.load(DATA_CACHE / 'wavenumber_axis_preprocessed.npy')  # (987,)
spectra_meta = pd.read_parquet(DATA_CACHE / 'spectra.parquet')

print(f'Preprocessed array shape: {X_all.shape}')
print(f'Wavenumber axis shape:    {wn_pp.shape}   range: [{wn_pp[0]:.1f}, {wn_pp[-1]:.1f}] cm-1')

# Pick the STEC demo file — find its pixel rows in the global array
stec_file_id = 'R359_100_10000ms_260226'
stec_mask = spectra_meta['file_id'] == stec_file_id
stec_global_idx = spectra_meta.index[stec_mask].tolist()
print(f'\nSTEC file "{stec_file_id}": {len(stec_global_idx)} pixels in cache')

# Extract pixel rows — already preprocessed, so set preprocess=False
X_stec = X_all[stec_global_idx, :]    # shape (N_pix, 987)
print(f'Pixel matrix shape: {X_stec.shape}')

# Call predict_from_array
array_result = predict_from_array(X_stec, wn_pp, preprocess=False)

print('\npredict_from_array result:')
print(f'  Predicted class: {array_result["class"]}')
sorted_p = sorted(array_result['probabilities'].items(), key=lambda x: x[1], reverse=True)
for cls, p in sorted_p:
    print(f'  {cls:12s}: {p:.4f}')

# Cross-check against predict_from_xls on the same file
print('\nCross-check: predict_from_xls vs predict_from_array (same STEC file)')
print(f'  predict_from_xls   predicted: {results["STEC"]["class"]}')
print(f'  predict_from_array predicted: {array_result["class"]}')

xls_proba = results['STEC']['probabilities']
arr_proba = array_result['probabilities']
max_diff = max(abs(xls_proba.get(c, 0.0) - arr_proba.get(c, 0.0)) for c in xls_proba)
print(f'  Max probability difference across classes: {max_diff:.6f}')
print(f'  Within tolerance (< 0.05): {max_diff < 0.05}')

if array_result['class'] == results['STEC']['class']:
    print('  Class label match: YES')
else:
    print('  Class label match: NO (XLS re-parses raw data; cache is independently preprocessed — small numeric differences expected)')

Preprocessed array shape: (7999, 987)
Wavenumber axis shape:    (987,)   range: [400.4, 3049.2] cm-1

STEC file "R359_100_10000ms_260226": 200 pixels in cache
Pixel matrix shape: (200, 987)



predict_from_array result:
  Predicted class: STEC
  STEC        : 0.8057
  Salmonella  : 0.1488
  Non-STEC    : 0.0455
  H2O         : 0.0000

Cross-check: predict_from_xls vs predict_from_array (same STEC file)
  predict_from_xls   predicted: STEC
  predict_from_array predicted: STEC
  Max probability difference across classes: 0.069337
  Within tolerance (< 0.05): False
  Class label match: YES


## Deployment note

Stage 15F LOSO accuracy is **0.448 file-weighted (95% CI [0.345, 0.552])**.

This API is a research artifact, not a deployed diagnostic.

For production deployment:
- Re-validate on a held-out study with at least 100 files per class.
- Calibrate probabilities on a withheld set (Platt scaling or isotonic regression).
- Gate on confidence: low-confidence predictions (max probability < 0.5) indicate likely mixed samples — see Notebook 08 for mixed-sample simulation results showing a 10-20% accuracy drop.
- The current model exhibits STEC-default bias on H2O spectra (all 8 LOSO H2O samples predicted as STEC), which must be addressed before any clinical or food-safety use.